In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.15 Orbital Angular Momentum and the Spherical Harmonics

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.15",
    title="Orbital Angular Momentum and the Spherical Harmonics",
    blurb="The algebra, made into shapes. When angular momentum comes from actual "
    "motion in space, its operators become derivatives on the sphere and its "
    "eigenstates become the spherical harmonics — the same s, p, and d shapes that "
    "label every atomic orbital. The abstract spectrum of the previous notebook "
    "reappears as functions one can plot, and a single physical demand, that the "
    "wavefunction not contradict itself after a full turn, quietly forbids the "
    "half-integers and leaves them to spin.",
    difficulty="advanced",
    estimate="160–195 min",
)

## Notebook overview

The previous notebook ([§6.14](angular-momentum-algebra.ipynb)) built angular momentum from nothing but an algebra, and the spectrum fell
out as abstract multiplets $|j,m\rangle$ — vectors in a $(2j+1)$-dimensional space, with no picture
attached. This notebook gives the picture. When the angular momentum is **orbital** — when it comes from
a particle actually moving through space, $\mathbf{L}=\mathbf{r}\times\mathbf{p}$ — the operators stop
being abstract matrices and become **derivatives on the sphere**, and their simultaneous eigenfunctions
become concrete functions of the two angles: the **spherical harmonics** $Y_l^m(\theta,\varphi)$.

Everything from [§6.14](angular-momentum-algebra.ipynb) reappears, now realized in calculus. Writing $\mathbf{p}=-i\hbar\nabla$ in spherical
coordinates turns $L_z$ into $-i\hbar\,\partial/\partial\varphi$ and $L^2$ into the angular part of the
Laplacian — the radial coordinate drops out entirely, so angular momentum acts only on *direction*. The
eigenvalue equations $L^2Y_l^m=\hbar^2 l(l+1)Y_l^m$ and $L_zY_l^m=\hbar m\,Y_l^m$ are the abstract
$j(j+1)$ and $m$ spectra of [§6.14](angular-momentum-algebra.ipynb), now satisfied by functions we can plot and integrate. The ladder
operators $L_\pm$ become differential operators that step between harmonics, and the spherical harmonics
are **orthonormal and complete** on the sphere — the angular analogue of the Fourier basis ([§6.9](position-representation.ipynb)), the
universal basis for *any* function of direction.

Then comes the twist the pure algebra could not see. The $L_z$ eigenfunctions carry the factor
$e^{im\varphi}$, and a wavefunction in real space must be **single-valued**: advancing $\varphi$ by
$2\pi$ returns to the same physical point, so $e^{2\pi i m}=1$, which forces $m$ — and hence $l$ — to be
an **integer**. The half-integers that the algebra of [§6.14](angular-momentum-algebra.ipynb) permitted have *no orbital realization*. They
survive only for **spin**, which has no position-space wavefunction and so escapes the argument entirely.
This is the deep reason spin is called "intrinsic": the algebra permits half-integers, geometry forbids
them for orbital motion, and spin is precisely what is left over.

Finally we connect to chemistry. The complex $Y_l^m$ are the $L_z$ eigenstates; their real combinations
are the familiar **orbital shapes** — $s$ (spherical), $p$ (two lobes), $d$ (cloverleaves) — the real
spherical harmonics first built in Volume III, reused here. Every atomic orbital ever drawn is one of
these functions, and they had to be these and no others, because the algebra of rotation and the geometry
of the sphere left no other choice.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — `scipy.special.sph_harm_y` for the harmonics, the angular
Laplacian by finite differences in $\theta$ and the periodic $\varphi$, and `numpy.trapezoid` with the
$\sin\theta$ measure for inner products on the sphere.

> **Conventions and method notes.** $\hbar=1$. We use the **current** SciPy API
> `scipy.special.sph_harm_y(n, m, theta, phi)` with $n=l$ the degree, $\theta\in[0,\pi]$ the **polar**
> (colatitude) angle, and $\varphi\in[0,2\pi)$ the **azimuth** — note the old `scipy.special.sph_harm`
> was *removed*, and both its name and its argument order ($\mathrm{sph\_harm}(m,l,\varphi,\theta)$) have
> changed. The operators $L_z,L^2,L_\pm$ are realized as **finite differences** on a $(\theta,\varphi)$
> grid; the $1/\sin\theta$ factors in $L^2$ and $L_\pm$ make the finite differences degrade **near the
> poles**, so we verify eigenvalue equations in the **bulk** ($\theta$ away from $0,\pi$) and say so.
> $L_z=-i\hbar\,\partial_\varphi$ has no such factor and is reliable everywhere. See Sakurai &
> Napolitano (§3.6); Griffiths (the angular equation); and Notebooks [§6.14](angular-momentum-algebra.ipynb) (the algebra/spectrum/ladder,
> here realized), [§6.6](pauli-uncertainty.ipynb) (incompatibility), [§6.9](position-representation.ipynb) (the basis-expansion idea), and Volume III (the real
> spherical harmonics and orbital visualization).

## Theory in brief

### Orbital angular momentum as a differential operator

For a particle moving in space, $\mathbf{L}=\mathbf{r}\times\mathbf{p}$ with $\mathbf{p}=-i\hbar\nabla$.
In spherical coordinates this becomes purely **angular**,

```{math}
:label: eq-orbital-operators
L_z=-i\hbar\,\frac{\partial}{\partial\varphi},\qquad
L^2=-\hbar^2\left[\frac{1}{\sin\theta}\frac{\partial}{\partial\theta}\Big(\sin\theta\,\frac{\partial}{\partial\theta}\Big)+\frac{1}{\sin^2\theta}\frac{\partial^2}{\partial\varphi^2}\right],
```

exactly the angular part of the Laplacian $\nabla^2$. The radial coordinate has dropped out: angular
momentum acts only on direction. These operators satisfy the same algebra $[L_i,L_j]=i\hbar\,
\varepsilon_{ijk}L_k$ as [§6.14](angular-momentum-algebra.ipynb), now realized in calculus.

### The spherical harmonics

The simultaneous eigenfunctions of $L^2$ and $L_z$ are the **spherical harmonics**,

```{math}
:label: eq-orbital-spherical-harmonics
L^2 Y_l^m=\hbar^2 l(l+1)\,Y_l^m,\qquad L_z Y_l^m=\hbar m\,Y_l^m,\qquad Y_l^m(\theta,\varphi)=N_{lm}\,P_l^m(\cos\theta)\,e^{im\varphi},
```

where $e^{im\varphi}$ carries the $L_z$ eigenvalue and the associated Legendre function $P_l^m(\cos\theta)$
gives the polar shape. These are the abstract $|j,m\rangle\to|l,m\rangle$ multiplets of [§6.14](angular-momentum-algebra.ipynb), now
functions on the sphere.

### Orthonormality and completeness

Under the inner product with the spherical area measure $\sin\theta\,d\theta\,d\varphi$,

```{math}
:label: eq-sphere-orthonormality
\langle Y_l^m\,|\,Y_{l'}^{m'}\rangle=\int_0^{2\pi}\!\!\int_0^\pi Y_l^{m\,*}\,Y_{l'}^{m'}\,\sin\theta\,d\theta\,d\varphi=\delta_{ll'}\delta_{mm'},
```

the spherical harmonics are **orthonormal**, and they are **complete**: any function on the sphere
expands as $\sum_{l,m}c_{lm}Y_l^m$ (the angular analogue of the basis expansions of [§6.1](complex-vector-spaces.ipynb) and [§6.9](position-representation.ipynb)). This is
why they are the universal angular basis.

### The ladder operators on the sphere

The raising and lowering operators $L_\pm=L_x\pm iL_y$ become

```{math}
:label: eq-orbital-ladder
L_\pm=\hbar\,e^{\pm i\varphi}\Big(\pm\frac{\partial}{\partial\theta}+i\cot\theta\,\frac{\partial}{\partial\varphi}\Big),\qquad L_\pm Y_l^m=\hbar\sqrt{l(l+1)-m(m\pm1)}\;Y_l^{m\pm1},
```

the abstract ladder of [§6.14](angular-momentum-algebra.ipynb), now a differential operator stepping between spherical harmonics — $L_+$
annihilates the top $Y_l^{\,l}$ and $L_-$ the bottom $Y_l^{-l}$.

### Why $l$ and $m$ are integers

The $L_z$ eigenfunctions carry $e^{im\varphi}$. Physical **single-valuedness** — the wavefunction must
return to itself when $\varphi$ advances by $2\pi$ — demands

```{math}
:label: eq-integer-restriction
e^{im(\varphi+2\pi)}=e^{im\varphi}\ \Longrightarrow\ e^{2\pi i m}=1\ \Longrightarrow\ m\in\mathbb{Z},
```

and the ladder then forces $l\in\{0,1,2,\dots\}$. So **orbital** angular momentum admits only integer
values: the half-integers the pure algebra allowed ([§6.14](angular-momentum-algebra.ipynb)) are excluded for orbital motion. They survive
only for **spin**, which has no position-space wavefunction and escapes the single-valuedness argument —
the deep reason spin is "intrinsic."

### The orbital shapes

The complex $Y_l^m$ are $L_z$ eigenstates; the **real** linear combinations $Y_l^m\pm Y_l^{-m}$ are the
familiar orbital shapes,

```{math}
:label: eq-orbital-shapes
l=0\ (s,\ \text{spherical}),\quad l=1\ (p,\ \text{two lobes}),\quad l=2\ (d,\ \text{cloverleaves}),\ \dots\qquad 2l+1\ \text{orbitals per shell},
```

the real spherical harmonics of Volume III, reused here. The labels $l=0,1,2,3$ are $s,p,d,f$; the count
$2l+1$ is the number of $m$ values. This is the angular skeleton of the periodic table.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import sph_harm_y

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED, BLUE = "#c1121f", "#1d4e89"

HBAR = 1.0

# the (theta, phi) grid: theta polar in [0, pi], phi azimuth in [0, 2pi) (periodic)
N_THETA, N_PHI = 300, 300
THETA = np.linspace(0.0, np.pi, N_THETA)
PHI = np.linspace(0.0, 2 * np.pi, N_PHI, endpoint=False)
DTH = THETA[1] - THETA[0]
DPH = PHI[1] - PHI[0]
TH, PH = np.meshgrid(THETA, PHI, indexing="ij")
SIN_TH = np.sin(TH)
# the "bulk" excludes a margin around the poles, where the 1/sinθ finite differences degrade
BULK = (TH > 0.20) & (TH < np.pi - 0.20)


def spherical_harmonic(l, m, theta, phi):
    """The spherical harmonic $Y_l^m(\\theta,\\varphi)$, via ``scipy.special.sph_harm_y``.

    Uses the **current** SciPy signature ``sph_harm_y(n=l, m, theta, phi)`` with $\\theta\\in[0,\\pi]$ the
    polar angle and $\\varphi\\in[0,2\\pi)$ the azimuth. (The old ``scipy.special.sph_harm`` was removed;
    both its name and its argument order — ``sph_harm(m, l, phi, theta)`` — changed.) The structure is
    $Y_l^m=N_{lm}P_l^m(\\cos\\theta)e^{im\\varphi}$ {eq}`eq-orbital-spherical-harmonics`.
    """
    return sph_harm_y(l, m, theta, phi)


def Lz_operator(field):
    """Apply $L_z=-i\\hbar\\,\\partial_\\varphi$ to a field on the $(\\theta,\\varphi)$ grid {eq}`eq-orbital-operators`.

    The $\\varphi$-derivative is a periodic central difference (``numpy.roll`` along the azimuth axis).
    Because $L_z$ has no $1/\\sin\\theta$ factor, it is accurate everywhere, including at the poles.
    """
    d_phi = (np.roll(field, -1, axis=1) - np.roll(field, 1, axis=1)) / (2 * DPH)
    return -1j * HBAR * d_phi


def L2_operator(field):
    """Apply the angular Laplacian $L^2$ to a field on the $(\\theta,\\varphi)$ grid {eq}`eq-orbital-operators`.

    Implements $L^2=-\\hbar^2[(1/\\sin\\theta)\\partial_\\theta(\\sin\\theta\\,\\partial_\\theta)+(1/\\sin^2
    \\theta)\\partial_\\varphi^2]$ by finite differences: ``numpy.gradient`` in $\\theta$ (non-periodic) and
    periodic central differences in $\\varphi$. The $1/\\sin\\theta$ factors degrade near the poles, so
    eigenvalue checks use the **bulk** (away from $\\theta=0,\\pi$).
    """
    d_th = np.gradient(field, DTH, axis=0)
    d2_phi = (
        np.roll(field, -1, axis=1) - 2 * field + np.roll(field, 1, axis=1)
    ) / DPH**2
    with np.errstate(divide="ignore", invalid="ignore"):
        term_theta = (1.0 / SIN_TH) * np.gradient(SIN_TH * d_th, DTH, axis=0)
        term_phi = (1.0 / SIN_TH**2) * d2_phi
    return -(HBAR**2) * (term_theta + term_phi)


def Lplus_operator(field):
    """Apply $L_+=\\hbar e^{i\\varphi}(\\partial_\\theta+i\\cot\\theta\\,\\partial_\\varphi)$ to a field {eq}`eq-orbital-ladder`.

    The differential realization of the raising operator on the sphere; like $L^2$ it carries a
    $\\cot\\theta$ that degrades near the poles, so it is verified in the bulk.
    """
    d_th = np.gradient(field, DTH, axis=0)
    d_phi = (np.roll(field, -1, axis=1) - np.roll(field, 1, axis=1)) / (2 * DPH)
    with np.errstate(divide="ignore", invalid="ignore"):
        cot = np.cos(TH) / SIN_TH
        result = HBAR * np.exp(1j * PH) * (d_th + 1j * cot * d_phi)
    return result


def sphere_inner(Y1, Y2):
    """The inner product $\\langle Y_1|Y_2\\rangle=\\int Y_1^* Y_2\\,\\sin\\theta\\,d\\theta\\,d\\varphi$ {eq}`eq-sphere-orthonormality`.

    Integrates over the $(\\theta,\\varphi)$ grid with the spherical area measure $\\sin\\theta$:
    ``numpy.trapezoid`` in $\\theta$ and a periodic sum $\\times d\\varphi$ in the azimuth.
    """
    integrand = np.conj(Y1) * Y2 * SIN_TH
    return np.trapezoid(np.sum(integrand, axis=1) * DPH, THETA)


def eigenvalue_in_bulk(L_field, field):
    """Read the eigenvalue of a (numerically applied) operator as the bulk ratio ``L_field / field``.

    Takes the median of $L_\\text{field}/\\text{field}$ over the bulk grid where $|\\text{field}|$ is not
    tiny, avoiding both the polar finite-difference error and division by near-zero amplitudes.
    """
    mask = BULK & (np.abs(field) > 0.05 * np.abs(field).max())
    return np.median((L_field[mask] / field[mask]).real)


def real_spherical_harmonic(l, m, theta, phi):
    """The **real** spherical harmonic (the orbital shape), built from ``sph_harm_y`` (Volume III).

    The familiar chemist's orbitals: $Y_l^0$ real for $m=0$, and for $m\\ne0$ the real/imaginary
    combinations $Y_l^{m,\\text{real}}\\propto P_l^{|m|}\\cos|m|\\varphi$ (or $\\sin|m|\\varphi$) that carry
    the azimuthal lobes. These are the $s,p,d,f$ shapes {eq}`eq-orbital-shapes`, reused from Volume III's
    multipole-expansion notebook.
    """
    if m == 0:
        return sph_harm_y(l, 0, theta, phi).real
    if m > 0:
        return np.sqrt(2.0) * (-1.0) ** m * sph_harm_y(l, m, theta, phi).real
    return np.sqrt(2.0) * (-1.0) ** m * sph_harm_y(l, -m, theta, phi).imag

## Exercise 1 — $L_z$ as a differential operator

Verify that the spherical harmonics are eigenfunctions of $L_z=-i\hbar\,\partial/\partial \varphi$
with eigenvalue $\hbar m$ {eq}`eq-orbital-operators`, {eq}`eq-orbital-spherical-harmonics`. The abstract
$J_z$ of [§6.14](angular-momentum-algebra.ipynb) is now a derivative in the azimuthal angle.

1. Build $Y_l^m$ on the $(\theta,\varphi)$ grid with `spherical_harmonic` (which wraps
   `scipy.special.sph_harm_y(l, m, theta, phi)`).
2. Apply $-i\hbar\,\partial_\varphi$ with the `Lz_operator` helper (a periodic central difference
   in $\varphi$).
3. Confirm the result is $\hbar m \,Y_l^m$ by reading the bulk ratio with `eigenvalue_in_bulk`.
4. Note the $e^{im\varphi}$ factor carries the eigenvalue — this is why $L_z$ is simply
   $-i\hbar\,\partial_\varphi$. Frame: $L_z$ realized in calculus.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    lz_ok,
    "the spherical harmonics are Lz eigenfunctions with eigenvalue ℏm (Lz = −iℏ∂/∂φ, finite difference)",
)

## Exercise 2 — $L^2$ as the angular Laplacian

Verify that the spherical harmonics are eigenfunctions of $L^2$ — the angular part of the
Laplacian — with eigenvalue $\hbar^2 l(l+1)$ {eq}`eq-orbital-spherical-harmonics`, the abstract $j(j+1)$
spectrum of [§6.14](angular-momentum-algebra.ipynb) now realized as a differential-operator eigenvalue.

1. Implement $L^2=-\hbar^2[(1/\sin\theta)\partial_\theta(\sin\theta\,\partial_\theta)+(1/
   \sin^2\theta)\partial_\varphi^2]$ (the `L2_operator` helper: `numpy.gradient` in $\theta$,
   periodic differences in $\varphi$).
2. Apply it to $Y_l^m$.
3. Confirm the eigenvalue is $\hbar^2 l(l+1)$ in the **bulk** with `eigenvalue_in_bulk` — near the
   poles the $1/\sin\theta$ finite differences degrade, so the check is restricted to $\theta$
   away from $0,\pi$.
4. Connect to [§6.14](angular-momentum-algebra.ipynb): this is the same $l(l+1)$ the algebra produced, now an eigenvalue of a
   derivative operator. Frame: the algebraic spectrum as calculus.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    l2_ok,
    "the spherical harmonics are L² eigenfunctions with eigenvalue ℏ²l(l+1) (the §6.14 spectrum realized as the angular Laplacian, verified in the bulk)",
)

## Exercise 3 — Orthonormality on the sphere

Verify that the spherical harmonics are **orthonormal** under the spherical inner product with the
$\sin\theta$ area measure, $\langle Y_l^m|Y_{l'}^{m'}\rangle=\delta_{ll'}\delta_{mm'}$
{eq}`eq-sphere-orthonormality`.

1. Use the spherical inner product $\langle Y|Y'\rangle=\int Y^*Y'\sin\theta\,d\theta\, d\varphi$
   (the `sphere_inner` helper, `numpy.trapezoid` with the $\sin\theta$ measure).
2. Compute $\langle Y_l^m|Y_l^m\rangle=1$ for several $(l,m)$ (normalization).
3. Compute $\langle Y_l^m|Y_{l'}^{m'} \rangle=0$ for different $(l,m)$ (orthogonality).
4. Note completeness: any function on the sphere expands in them. Frame: the universal angular
   basis — the basis idea of [§6.1](complex-vector-spaces.ipynb)/[§6.9](position-representation.ipynb), on the sphere.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    np.array(norms + cross),
    np.array([1.0] * len(norms) + [0.0] * len(cross)),
    "the spherical harmonics are orthonormal on the sphere under the sinθ measure",
    atol=1e-2,
)

## Exercise 4 — The ladder on the sphere

Verify that the raising operator $L_+$ — now a differential operator on the sphere — steps $Y_l^0$
up to $Y_l^1$ with the algebraic coefficient $\hbar\sqrt{l(l+1)-m(m+1)}$, and annihilates the top
state $Y_l^{\,l}$ {eq}`eq-orbital-ladder`.

1. Implement $L_+=\hbar e^{i\varphi}(\partial_\theta+i\cot\theta\,\partial_\varphi)$ (the
   `Lplus_operator` helper).
2. Apply it to $Y_l^0$.
3. Confirm the result is proportional to $Y_l^1$ with coefficient
   $\hbar\sqrt{l(l+1)-0}=\hbar\sqrt{l(l+1)}$ (the bulk ratio against $Y_l^1$).
4. Confirm $L_+$ annihilates the top rung $Y_l^{\,l}$ (its bulk norm is negligible). Frame: the
   [§6.14](angular-momentum-algebra.ipynb) ladder, now a differential operator stepping between functions.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    ladder_ok and top_ok,
    "L₊ raises m with coefficient ℏ√(l(l+1)−m(m+1)) and annihilates the top rung Yₗˡ — the §6.14 ladder as a differential operator",
)

## Exercise 5 — Why $l$ and $m$ are integers

Show that physical **single-valuedness** forces $m$, and hence $l$, to be **integers** — excluding
the half-integer angular momenta that the pure algebra of [§6.14](angular-momentum-algebra.ipynb) permitted
{eq}`eq-integer-restriction`. This is a constraint the algebra alone could not see.

1. Note the $L_z$ eigenfunctions carry $e^{im\varphi}$.
2. Require single-valuedness under a full turn: $e^{im(\varphi+2\pi)}=e^{im\varphi}$, i.e.
   $e^{2\pi im}=1$.
3. Confirm with `numpy.exp` that this holds **only** for integer $m$ (test
   $m=\tfrac12,1,\tfrac32,2,\tfrac52$).
4. Conclude orbital angular momentum has only integer $l$; the half-integers belong to **spin**,
   which has no position-space wavefunction and so escapes the argument. Frame: the algebra
   permits half-integers, geometry forbids them for orbital motion, and spin is what remains.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    integer_only,
    "single-valuedness (e^{2πim}=1) holds only for integer m, so orbital l,m are integers — the half-integers of §6.14 are excluded and belong to spin",
)

## Exercise 6 — The orbital shapes: from $Y_l^m$ to $s,p,d$

Build the **real** spherical harmonics and identify the familiar orbital shapes — $s$ (spherical),
$p$ (lobed), $d$ (cloverleaves) — confirming there are $2l+1$ per shell {eq}`eq-orbital-shapes`.
The $L^2,L_z$ eigenfunctions are the orbitals of chemistry.

1. Recall the complex $Y_l^m$ are $L_z$ eigenstates; form the **real** combinations (the
   `real_spherical_harmonic` helper, reused from Volume III).
2. Check the structural facts: $s$ ($l=0$) is constant in angle (spherical), $p_z$ ($l=1,m=0$) is
   $\propto\cos\theta$ (sign-flips across the equator), and there are $2l+1$ functions for each
   $l$.
3. Plot the $s,p,d$ real orbitals as 3-D surfaces (radius $\propto|Y^{\text{real}}|$, color by
   sign).
4. Note the labels $s,p,d,f$ for $l=0,1,2,3$. Frame: the spherical harmonics are the angular
   skeleton of the periodic table.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    shapes_ok,
    "the real spherical harmonics are the s/p/d orbital shapes (s spherical, p_z lobed with a sign change, 2l+1 per shell) — the atomic orbitals (Volume III)",
)

In [ ]:
# (solution hidden on the public site)


## The phase behind the shapes

The gallery above is deceptively still. Each surface is the *modulus* $|Y_l^m|$, but $Y_l^m$ is a
complex **wave**, and the lobe pictures quietly throw away its phase $e^{im\varphi}$. Worse, a
stationary state is not actually frozen: it evolves in time as $\psi(t)=Y_l^m\,e^{-iEt/\hbar}$ (the
time-evolution of [§6.7](time-evolution.ipynb)). Because $e^{-iEt/\hbar}$ is a *global* phase, the probability density
$|\psi(t)|^2=|Y_l^m|^2$ stands perfectly still — nothing observable changes — yet the phase itself keeps
turning underneath.

Colouring each fixed lobe by its running phase $\arg(Y_l^m e^{-iEt})$ makes that hidden turning visible,
and with it the physical meaning of the quantum number $m$. For $m\ne0$ the colour **circulates around
the $z$-axis**, winding $m$ times: a *running* wave that carries angular momentum $\hbar m$ about $z$ —
this is what "magnetic quantum number" means. For $m=0$ the real harmonic simply **pulses in place**, a
standing wave with no azimuthal current. The illustration below (not an exercise) animates one full
period; watch the colour circulate for $m\ne0$ and pulse for $m=0$, while every shape holds still.

In [ ]:
# (solution hidden on the public site)


```{admonition} With your assistant
:class: tip
The gallery machinery of Exercise 6 — meshgrids, 3-D axes, colormaps, view
angles — is visualization plumbing, and delegating it outright is exactly what
the plumbing deserves. The physics cells stay yours: whatever your assistant
renders, confirm the surfaces it drew are $|Y_l^m|$ on the correct
$(\theta,\phi)$ grid by checking one value you can compute by hand —
$|Y_0^0| = 1/\sqrt{4\pi}$ everywhere is enough to catch a swapped axis. The
check is yours.
```

## Exercise 7 — A superposition on the sphere and its angular-momentum statistics *(student)*

Given an angular wavefunction that is a superposition of spherical harmonics, find the
probabilities of measuring each $(l,m)$ and the expectation values $\langle L_z\rangle$ and
$\langle L^2 \rangle$, applying the Born rule ([§6.5](postulates.ipynb)) in the angular basis
{eq}`eq-orbital-spherical-harmonics`, {eq}`eq-sphere-orthonormality`.

1. Build $\psi(\theta,\varphi)=\sum c_{lm}Y_l^m$ with given coefficients and normalize it with
   `sphere_inner` (since the $Y_l^m$ are orthonormal, $\langle\psi|\psi\rangle=\sum|c_{lm}|^2$).
2. The probability of measuring $(l,m)$ is $|c_{lm}|^2$ (the Born rule, [§6.5](postulates.ipynb), in the angular
   basis).
3. Compute $\langle L_z\rangle=\sum|c_{lm}|^2\hbar m$ and $\langle
   L^2\rangle=\sum|c_{lm}|^2\hbar^2 l(l+1)$.
4. Verify $\langle L_z\rangle$ directly as the operator expectation $\langle\psi|L_z|\psi\rangle$
   (the pole-free $L_z$, integrated over the full sphere). Frame: the postulates of Movement I,
   applied to angular momentum.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    born_ok,
    "the Born rule on the sphere: the probabilities |cₗₘ|² sum to 1 and ⟨Lz⟩=Σ|cₗₘ|²ℏm matches the direct operator expectation ⟨ψ|Lz|ψ⟩",
)

## Exercise 8 — The shapes of angular momentum

When angular momentum comes from real motion through space, the operators we built abstractly in [§6.14](angular-momentum-algebra.ipynb)
become **derivatives on the sphere**, and their eigenstates become the **spherical harmonics** — the
very $s$, $p$, and $d$ shapes that organize the periodic table. The abstract spectrum $\hbar^2 l(l+1)$,
$\hbar m$ reappeared as functions we could plot and integrate; the ladder became a differential operator
stepping between them; and a single physical demand — that a wavefunction not contradict itself after a
full turn — quietly excluded the half-integers, handing them to spin.

**This exercise (synthesis).** No new computation: the realization is the result. The half-integers of
[§6.14](angular-momentum-algebra.ipynb) were not wrong; they were simply not *orbital*. Single-valuedness on the sphere is a constraint the
pure algebra could not feel, and it is exactly what separates the integer orbital angular momenta — which
have a position-space wavefunction $Y_l^m(\theta,\varphi)$ — from the half-integer spins, which do not.
With the angular problem now solved once and for all, the next notebook ([§6.16](central-potentials-3d.ipynb)) attacks the full
three-dimensional Schrödinger equation: separate the angular part (these spherical harmonics) from the
radial part, and any central-force problem collapses to a one-dimensional radial equation of the kind we
already solved in [§6.10](schrodinger-on-a-computer.ipynb)–[§6.11](bound-states-1d.ipynb). That is the road to hydrogen ([§6.17](hydrogen-atom.ipynb)).

Every atomic orbital you have ever seen drawn is one of these functions. The chemist's $s$, $p$, and $d$
are not mnemonics — they are the eigenfunctions of $L^2$ and $L_z$, and they had to be these and nothing
else, because the algebra of rotation and the geometry of the sphere left no other choice.

## Notebook summary

The algebra of [§6.14](angular-momentum-algebra.ipynb), realized as calculus on the sphere — the second notebook of Movement III.

- **Orbital operators** {eq}`eq-orbital-operators`: $L_z=-i\hbar\,\partial_\varphi$ and $L^2$ the angular
  Laplacian — the radial coordinate drops out, so angular momentum acts only on direction.
- **The spherical harmonics** {eq}`eq-orbital-spherical-harmonics`: $L^2Y_l^m=\hbar^2 l(l+1)Y_l^m$, $L_zY_l^m=
  \hbar m Y_l^m$ (verified on a grid via `scipy.special.sph_harm_y`) — the [§6.14](angular-momentum-algebra.ipynb) spectrum as functions.
- **Orthonormality** {eq}`eq-sphere-orthonormality`: $\langle Y_l^m|Y_{l'}^{m'}\rangle=\delta_{ll'}
  \delta_{mm'}$ under the $\sin\theta$ measure — the universal, complete angular basis ([§6.1](complex-vector-spaces.ipynb)/[§6.9](position-representation.ipynb)).
- **The ladder** {eq}`eq-orbital-ladder`: $L_\pm$ as differential operators stepping between harmonics,
  with coefficient $\hbar\sqrt{l(l+1)-m(m\pm1)}$, annihilating the top/bottom rung.
- **Integers only** {eq}`eq-integer-restriction`: single-valuedness $e^{2\pi im}=1$ forces $l,m\in
  \mathbb{Z}$ — the half-integers of [§6.14](angular-momentum-algebra.ipynb) are excluded for orbital motion and belong to spin.
- **The orbital shapes** {eq}`eq-orbital-shapes`: the real $Y_l^m$ are the $s,p,d,f$ orbitals, $2l+1$ per
  shell (Volume III) — the angular skeleton of the periodic table.

The abstract multiplets became plottable functions, and geometry quietly forbade the half-integers. The
angular problem is solved; the radial problem is next.

## Outlook

- **The three-dimensional Schrödinger equation ([§6.16](central-potentials-3d.ipynb))**: separation of variables — the angular part is
  solved (these spherical harmonics), the radial part becomes a 1-D equation of the [§6.10](schrodinger-on-a-computer.ipynb)–[§6.11](bound-states-1d.ipynb) kind.
- **The hydrogen atom ([§6.17](hydrogen-atom.ipynb))**: the Coulomb radial equation meeting these angular solutions — the crown
  jewel of the volume.
- **Spin ([§6.18](spin-magnetic.ipynb)) and the addition of angular momenta ([§6.19](addition-angular-momenta.ipynb))**: the half-integers excluded here, and their
  coupling to orbital motion.
- **Selection rules and the shapes of spectral lines** (a horizon; time-dependent transitions, [§6.24](time-dependent-perturbation.ipynb)).
- **Cross-reference** [§6.14](angular-momentum-algebra.ipynb) (the algebra, spectrum, and ladder, here realized), [§6.6](pauli-uncertainty.ipynb) (incompatibility),
  [§6.5](postulates.ipynb) (the Born rule), Volume III (the real spherical harmonics), and forward to [§6.16](central-potentials-3d.ipynb), [§6.17](hydrogen-atom.ipynb), [§6.18](spin-magnetic.ipynb).

In [ ]:
from ecp.style import footer

footer()